<a href="https://colab.research.google.com/github/santidrelew/tp-mpg/blob/main/notebooks/03_OLS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3 - Regresión con OLS

In [7]:
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    !git clone https://github.com/santidrelew/tp-mpg.git
    %cd tp-mpg

Cloning into 'tp-mpg'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 123 (delta 61), reused 60 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 13.44 MiB | 16.56 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/tp-mpg/tp-mpg/tp-mpg


In [8]:
# Importamos todas las librerías necesarias
from pathlib import Path
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.formula.api as smf


In [14]:
# Cargamos los datasets
PROJECT_DIR = Path().resolve() if EN_COLAB else Path().resolve().parent
DATA_BASE_DIR = PROJECT_DIR / "data" / "procesado"

df = pd.read_csv(DATA_BASE_DIR / "milan_airbnb_filtrado.csv")
df_muestra = pd.read_csv(DATA_BASE_DIR / "milan_airbnb_muestra.csv")

Saving milan_airbnb_filtrado.csv to milan_airbnb_filtrado.csv


In [46]:
TARGET = "log_daily_price"
FEATURES = [c for c in df.columns if c != TARGET]
CATEGORICAS = ["neighbourhood_cleansed"]

Creamos una unica funcion para usar en los 4 casos y no tener que repetir codigo:

In [47]:
def construir_formula(target: str, variables: list, categoricas: list) -> str:
    terminos = [f"C({v})" if v in categoricas else v for v in variables]
    return f"{target} ~ " + " + ".join(terminos)


def calcular_metricas(modelo, df_data: pd.DataFrame, target: str) -> dict:
    y_true = df_data[target]
    y_pred = modelo.predict(df_data)

    return {
        "N Variables": int(modelo.df_model),
        "R2": r2_score(y_true, y_pred),
        "R2 Ajustado": modelo.rsquared_adj,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
    }

Primero, hacemos la regresion con todas las variables EDA de nuestro dataset entero:

In [48]:
formula_completa = construir_formula(TARGET, FEATURES, CATEGORICAS)
modelo_completo = smf.ols(formula=formula_completa, data=df).fit()

print(modelo_completo.summary())

                            OLS Regression Results                            
Dep. Variable:        log_daily_price   R-squared:                       0.395
Model:                            OLS   Adj. R-squared:                  0.393
Method:                 Least Squares   F-statistic:                     183.5
Date:                Mon, 09 Nov 2026   Prob (F-statistic):               0.00
Time:                        04:43:39   Log-Likelihood:                -5320.0
No. Observations:                9297   AIC:                         1.071e+04
Df Residuals:                    9263   BIC:                         1.095e+04
Df Model:                          33                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

En este analisis, podemos ver que las siguientes 6 variables no aportan evidencia estadística suficiente para rechazar la hipotesis nula y sus intervalos de confianza cruzan el cero:
* minimum_nights (p = 0.173)
* Washer (p = 0.168)
* Paid_parking_on_premises (p = 0.893)
* Long_term_stays_allowed (p = 0.231)
* Smoking_allowed (p = 0.225)
* check_in_24_hour (p = 0.258)

Ahora, haremos una nueva regresion sobre el dataset EDA completo pero omitiendo estas variables:

In [49]:
variables_no_significativas = ["minimum_nights", "Washer", "Paid_parking_on_premises", "Long_term_stays_allowed", "Smoking_allowed", "check_in_24_hour"]
variables_significativas = [cols for cols in FEATURES if cols not in variables_no_significativas]
formula_reducida_significativa = construir_formula(TARGET, variables_significativas, CATEGORICAS)
modelo_reducido_significativo = smf.ols(formula=formula_reducida_significativa, data=df).fit()

print(modelo_reducido_significativo.summary())

                            OLS Regression Results                            
Dep. Variable:        log_daily_price   R-squared:                       0.395
Model:                            OLS   Adj. R-squared:                  0.393
Method:                 Least Squares   F-statistic:                     223.9
Date:                Mon, 09 Nov 2026   Prob (F-statistic):               0.00
Time:                        04:43:39   Log-Likelihood:                -5324.4
No. Observations:                9297   AIC:                         1.070e+04
Df Residuals:                    9269   BIC:                         1.090e+04
Df Model:                          27                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

Ahora vamos a hacer el mismo procedimiento con la muestra seleccionada:

In [50]:
modelo_completo_muestra = smf.ols(formula=formula_completa, data=df_muestra).fit()

print(modelo_completo_muestra.summary())

                            OLS Regression Results                            
Dep. Variable:        log_daily_price   R-squared:                       0.432
Model:                            OLS   Adj. R-squared:                  0.406
Method:                 Least Squares   F-statistic:                     16.49
Date:                Mon, 09 Nov 2026   Prob (F-statistic):           3.13e-67
Time:                        04:43:39   Log-Likelihood:                -364.58
No. Observations:                 750   AIC:                             797.2
Df Residuals:                     716   BIC:                             954.2
Df Model:                          33                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

Vemos que las variables no significativas son muchas mas que en el modelo entrenado sobre el dataset completo:

* (ESCRIBIR CADA VARIABLE)

Ahora, haremos una nueva regresion sobre la muestra pero omitiendo estas variables:

In [51]:
variables_significativas_muestra = ["neighbourhood_cleansed", "host_total_listings_count", "accommodates", "bathrooms", "bedrooms", "number_of_reviews", "review_scores_rating", "WiFi"]
formula_reducida_significativa_muestra = construir_formula(TARGET, variables_significativas_muestra, CATEGORICAS)
modelo_reducido_significativo_muestra = smf.ols(formula=formula_reducida_significativa_muestra, data=df_muestra).fit()

print(modelo_reducido_significativo_muestra.summary())

                            OLS Regression Results                            
Dep. Variable:        log_daily_price   R-squared:                       0.414
Model:                            OLS   Adj. R-squared:                  0.402
Method:                 Least Squares   F-statistic:                     34.50
Date:                Mon, 09 Nov 2026   Prob (F-statistic):           7.88e-75
Time:                        04:43:39   Log-Likelihood:                -376.49
No. Observations:                 750   AIC:                             785.0
Df Residuals:                     734   BIC:                             858.9
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept   

Ahora comparemos las 4 regresiones que hicimos:

In [52]:
res_completo_pob = calcular_metricas(modelo_completo, df, TARGET)
res_reducido_pob = calcular_metricas(modelo_reducido_significativo, df, TARGET)
res_completo_mue = calcular_metricas(modelo_completo_muestra, df_muestra, TARGET)
res_reducido_mue = calcular_metricas(modelo_reducido_significativo_muestra, df_muestra, TARGET)

tabla_comparativa = pd.DataFrame(
    [
        res_completo_pob,
        res_reducido_pob,
        res_completo_mue,
        res_reducido_mue,
    ],
    index=[
        "1. Poblacion (Modelo completo)",
        "2. Población (Modelo significativo)",
        "3. Muestra (Modelo completo)",
        "4. Muestra (Modelo significativo)",
    ],
)

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
tabla_comparativa

,N Variables,R2,R2 Ajustado,MAE,RMSE
1. Poblacion (Modelo completo),33,0.3953,0.3932,0.3071,0.4288
2. Población (Modelo significativo),27,0.3947,0.3930,0.3072,0.4290
3. Muestra (Modelo completo),33,0.4318,0.4057,0.2981,0.3934
4. Muestra (Modelo significativo),15,0.4135,0.4015,0.3018,0.3997


* Tanto el MAE como el RMSE permanecen inalterados en el tercer decimal al reducir las variables. Quitar variables no significativas no degrad0 la precision predictiva.   

* En los cuatro modelos, el RMSE es mayor que el MAE, lo cual demuestra la presencia de alquileres con errores extremos (propiedades atípicas o fuera de mercado que el modelo subestima o sobreestima fuertemente).

* Como la variable dependiente esta expresada en escala logaritmica, un MAE de 0.30 indica que las predicciones del modelo se desvian, en promedio, aproximadamente un 30% respecto al valor real de la publicacion.  

* Poblacion (1 vs. 2): Al pasar de 33 a 27 variables, el R2 cayo de 0.3953 a 0.3947 (una baja insignificante de 0.06), mientras que el R2 ajustado se mantiene casi identico (0.3932 vs 0.3930). Esto comprueba que las 6 variables eliminadas aportaban unicamente ruido aleatorio y nula capacidad explicativa real.
* Muestra (3 vs. 4): Se eliminaron 18 variables (más del 50% de los predictores). Aun asi, el R2 ajustado solo vario de 0.4057 a 0.4015 (menos de medio punto porcentual). El modelo reducido conserva el 99% de la potencia explicativa ajustada del modelo saturado con menos de la mitad de las variables.

Comparando la muestra con el total del dataset EDA, podemos decir que:
* En la población de 9297 propiedades, casi cualquier efecto pequeño resulta estadisticamente significativo. En la muestra de 750, los errores estandar son mayores y las variables como amenities específicas (cafetera, plancha, ascensor) pierden significancia.

Las variables que se sostuvieron como significativas tanto en la poblacion como en la muestra son el nucleo explicativo fundamental del precio de un Airbnb en Milan:
* Ubicacion: neighbourhood_cleansed
* Capacidad del alojamiento: accommodates, bathrooms, bedrooms
* Reputacion y perfil del host: host_total_listings_count, number_of_reviews, review_scores_rating.
* Conectividad: WiFi.